# Rover on Rough Terrain — Planning Challenge

A point rover lives on a 2D arena. It must drive from **start** to **goal**.

You are given:
- `field` — a `(H, W, 3)` terrain map with 3 channels you can see:
  - **ch 0 — rocks**: hard obstacles (lethal, must avoid)
  - **ch 1 — mud**: continuous 0..1
  - **ch 2 — slope**: continuous 0..1
- `demos` — a handful of **expert trajectories** (lists of `(x, y)` points).
  Each demo runs between its *own* start/goal pair (NOT necessarily yours), so
  no single demo is the answer — they're there to reveal the terrain cost.
- `start`, `goal` — the 2D points YOUR path must connect (`x = column`, `y = row`).

**Motion model:** the rover is a point moving in continuous `(x, y)` space. Your
planner may return any polyline of waypoints; moves are not restricted to a
4-connected or 8-connected grid. For example, a grid-search implementation may
choose 8-connected neighbors and a sampling planner may produce arbitrary
diagonal segments. The metric samples along every segment, so a move cannot
skip over a rock or expensive terrain just by using sparse waypoints.

The experts know something you don't: the *true* traversal cost of the terrain.
Your job is to produce a path that reaches the goal **and** is about as
terrain-cheap as the experts'. A path that only dodges rocks will reach the
goal but cost too much — look at the demos to figure out what else to avoid.

**Metric (pre-built, do not edit):** a path passes if it
(1) starts near start, ends near goal, (2) never touches a rock,
(3) stays within the arena bounds (leaving the map fails),
(4) is traversable under your submitted cost map, and
(5) has terrain cost `<= expert_cost * tol`.

Work through the TODOs. You can stop at the classical solution or push on to
the learned one; partial progress is fine. Talk through your choices as you go.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os, sys

# Clone the data repo when running on Colab (no-op if files already exist).
if not os.path.exists('field.npy'):
    import subprocess
    subprocess.run(
        ['git', 'clone', '--depth', '1',
         'https://github.com/vvanirudh/planning_challenge.git', '_data'],
        check=True)
    os.chdir('_data')

field = np.load('field.npy')                          # (H, W, 3)
demos = np.load('demos.npy', allow_pickle=True)       # object array of (T,2)
meta  = np.load('meta.npz')
start, goal = meta['start'], meta['goal']
H, W = field.shape[:2]
ROCK, MUD, SLOPE = 0, 1, 2
print('field', field.shape, '| demos', len(demos),
      '| start', start, 'goal', goal)

In [ ]:
def show(paths=None, cost_map=None, title=''):
    fig, ax = plt.subplots(figsize=(6, 6))
    if cost_map is not None:
        ax.imshow(cost_map, origin='lower', cmap='viridis')
    else:
        # rocks black, mud red-ish, slope blue-ish
        rgb = np.zeros((H, W, 3))
        rgb[..., 0] = field[..., MUD]
        rgb[..., 2] = field[..., SLOPE]
        rgb[field[..., ROCK] > 0.5] = [0, 0, 0]
        ax.imshow(rgb, origin='lower')
    for d in demos:
        d = np.asarray(d)
        ax.plot(d[:, 0], d[:, 1], '0.7', lw=1, alpha=.7)
    if paths is not None:
        if isinstance(paths, np.ndarray) and paths.ndim == 2:
            paths = [paths]
        for p in paths:
            p = np.asarray(p)
            ax.plot(p[:, 0], p[:, 1], 'lime', lw=2)
    ax.plot(*start, 'wo', ms=8); ax.plot(*goal, 'w*', ms=14)
    ax.set_title(title); ax.set_xlim(0, W); ax.set_ylim(0, H)
    plt.show()

show(title='terrain (red=mud, blue=slope, black=rock) + expert demos (gray)')

In [ ]:
# ============================ PRE-BUILT METRIC ============================
# Do not edit. evaluate(path, cost_map) -> PASS/FAIL.
# The true per-step terrain cost is shipped as an opaque grid in meta.npz
# (meta['true_cost']); recovering what drives it from the demos is the point of
# the challenge, so don't peek at it -- just call evaluate(path).
EXPERT_COST = float(meta['expert_cost'])
TOL = float(meta['tol'])
_TRUE_COST = meta['true_cost']                 # (H,W) opaque scoring grid
_ROCK = float(meta['rock_sentinel'])           # cells >= this are rock
_STEP = 0.5                                    # path-integral resampling (cells)

def evaluate(path, cost_map):
    path = np.asarray(path, float)
    cost_map = np.asarray(cost_map, float)
    if cost_map.shape != (H, W):
        raise ValueError(f'cost_map must have shape {(H, W)}, got {cost_map.shape}')
    ok_start = np.linalg.norm(path[0] - start) < 5
    ok_goal  = np.linalg.norm(path[-1] - goal) < 5
    # Resample the path to ~uniform _STEP spacing and integrate cost along it,
    # so the score doesn't depend on how many waypoints you used (and a segment
    # crossing a rock between waypoints is still caught).
    seg = np.linalg.norm(np.diff(path, axis=0), axis=1)
    L = float(seg.sum())
    if L < 1e-9:
        return False
    s = np.concatenate([[0.0], np.cumsum(seg)])
    n = max(2, int(L / _STEP) + 1)
    u = np.linspace(0, L, n)
    xs = np.interp(u, s, path[:, 0]); ys = np.interp(u, s, path[:, 1])
    hit_rock = False
    model_feasible = True
    out_of_bounds = False
    cost = 0.0
    for x, y in zip(xs, ys):
        rx, ry = round(x), round(y)
        if not (0 <= rx <= W - 1 and 0 <= ry <= H - 1):
            out_of_bounds = True   # leaving the arena is infeasible (inf cost)
            continue
        c, r = int(rx), int(ry)
        if not np.isfinite(cost_map[r, c]): model_feasible = False
        cell = _TRUE_COST[r, c]
        if cell >= _ROCK: hit_rock = True
        else: cost += cell * _STEP
    bar = EXPERT_COST * TOL
    success = bool(ok_start and ok_goal and model_feasible
                   and not hit_rock and not out_of_bounds and cost <= bar)
    print(f"start_ok={ok_start} goal_ok={ok_goal} "
          f"model_feasible={model_feasible} rock_hit={hit_rock} "
          f"out_of_bounds={out_of_bounds} "
          f"cost={cost:.1f} (bar={bar:.1f}) -> {'PASS' if success else 'FAIL'}")
    return success

## TODO 1 — a cost over states (start classical)

In [ ]:
# ---- TODO 1: a cost over states ------------------------------------------
# Return a per-cell traversal cost map of shape (H, W). Cells the rover should
# avoid should be expensive; rocks should be ~infinite (impassable).
# A trivial uniform placeholder is provided so everything runs. Improve it:
# at minimum make rocks impassable; optionally guess mud/slope penalties, then
# run the planner below and see why this isn't enough.
def cost_map_hand():
    '''Hand-designed (H,W) cost. No demos used.'''
    cm = np.ones((H, W), float)   # uniform placeholder -- everything costs 1
    # TODO: make rocks impassable; optionally guess mud/slope penalties.
    return cm

## TODO 2 — sampling rollout planner

In [ ]:
# ---- TODO 2: a sampling-based rollout planner ---------------
# Plan a path start->goal that is cheap under a given (H,W) cost_map.
# Suggested scheme (you may do something else):
#   - keep a current trajectory of T waypoints from start toward goal
#   - sample K noisy candidate rollouts around it
#   - score each by sum of cost_map along it (+ a goal-reaching term)
#   - update toward the low-cost samples (softmax / take-best)
# Keep it on CPU/numpy first. We'll talk about batching/GPU after.
def plan(cost_map, K=256, T=60, iters=30, seed=0):
    '''Return an (T,2) path of (x,y) points from start to goal.'''
    rng = np.random.default_rng(seed)
    # Trivial placeholder: a straight line start->goal. It ignores cost_map
    # entirely (so it drives through whatever is in the way). Replace with a
    # sampling rollout that actually uses cost_map.
    # TODO: implement the sampling rollout loop.
    return np.linspace(start, goal, T)

In [ ]:
# Run the hand-cost classical solution:
path = plan(cost_map_hand())
show(path, title='classical: hand cost')
evaluate(path, cost_map_hand())   # expect: reaches goal, but terrain cost over the bar

## TODO 3 — learn the cost from demos

In [ ]:
# ---- TODO 3: learn the cost from the expert demos ------------------------
# The experts detour around something the rock map doesn't show. Use the demos
# to build a cost map that explains their behavior, then re-plan.
# Many valid approaches — pick one and justify it:
#   - visitation: cells experts traverse are cheap, cells they avoid costly
#   - feature regression / IRL: fit weights w over [mud, slope] so expert
#     paths are low-cost relative to alternatives
#   - nearest-demo / warping
def cost_map_learned(field, demos):
    '''Return an (H,W) cost map inferred from demos. Keep rocks impassable.'''
    # Placeholder: same uniform cost as the hand version (ignores demos).
    # Replace this with something that USES the demos.
    # TODO: your approach here
    return cost_map_hand()

In [ ]:
# Run the learned solution:
cm = cost_map_learned(field, demos)
show(cost_map=cm, title='learned cost map')
path = plan(cm)
show(path, title='learned: planned path')
evaluate(path, cm)   # goal: PASS

### Bonus / discussion — make it fast

Your planner rolls out `K` trajectories every iteration. On a real robot we
replan in a closed loop at, say, 50 Hz with `K` in the thousands.

- **Vectorize** the rollout: evaluate all `K` candidates without a Python loop
  (batched numpy, or move the cost lookup + scoring to `torch` on GPU).
- Be ready to discuss: memory layout of the `K × T × 2` buffer, how `K` trades
  off against horizon `T`, host↔device transfer / sync cost, and what your
  per-replan time budget buys you at 50 Hz.

In [ ]:
# ---- BONUS TODO: vectorized / GPU rollout --------------------------------
# Re-implement the per-iteration rollout scoring without a Python loop over K.
# numpy: index cost_map with a (K, T) array of flattened cell ids.
# torch: keep cost_map + samples on cuda; one gather + sum.
# (Optional — sketch it even if you don't finish.)